In [2]:
import numpy as np

# Introduction to numerical methods

János Török

# Class 3: Linear algebra 2.

## System of linear equations

The general form is:
$$\begin{cases}
a_{11}x_1 + a_{12}x_2 + \dots + a_{1n}x_n = b_1 \\
a_{21}x_1 + a_{22}x_2 + \dots + a_{2n}x_n = b_2 \\
\vdots\\
a_{m1}x_1 + a_{m2}x_2 + \dots + a_{mn}x_n = b_m,
\end{cases}$$
where $x_1, x_2,\dots,x_n$ are the unknowns, $a_{ij}$ is the coefficient of $x_j$ in equation $i$, and $b_1,b_2,\dots,b_m$ are the constant terms.

### In matrix form
$$A\mathbf{x}=\mathbf{b},$$
where
$$A=
\begin{pmatrix}
a_{11} & a_{12} & \cdots & a_{1n} \\
a_{21} & a_{22} & \cdots & a_{2n} \\
\vdots & \vdots & \ddots & \vdots \\
a_{m1} & a_{m2} & \cdots & a_{mn}
\end{pmatrix},\quad
\mathbf{x}=
\begin{pmatrix}
x_1 \\
x_2 \\
\vdots \\
x_n
\end{pmatrix},\quad
\mathbf{b}=
\begin{pmatrix}
b_1 \\
b_2 \\
\vdots \\
b_m
\end{pmatrix}.$$

Here $A$ is the coefficient matrix, $\mathbf{x}$ is the vector of unknowns, and $\mathbf{b}$ contains the right-hand sides.

### Gauss elimination

First we create the augmented matrix $(A|b)$. Then we try to make the values below the diagonal zero; when convenient, we also scale each pivot to one. Allowed steps are:
* Swapping two rows,
* Multiplying a row by a nonzero number,
* Adding a multiple of one row to another row.

Example:

$$\begin{pmatrix}
1 & 3 & 1 & 9 \\
1 & 1 & -1 & 1 \\
3 & 10 & 5 & 32
\end{pmatrix}\to
\begin{pmatrix}
1 & 3 & 1 & 9 \\
0 & -2 & -2 & -8 \\
0 & 1 & 2 & 5
\end{pmatrix}\to
\begin{pmatrix}
1 & 3 & 1 & 9 \\
0 & -2 & -2 & -8 \\
0 & 0 & 1 & 1
\end{pmatrix}\to
\begin{pmatrix}
1 & 3 & 1 & 9 \\
0 & 1 & 1 & 4 \\
0 & 0 & 1 & 1
\end{pmatrix}$$

The steps:
1. The first row is subtracted from the second row, and three times the first row is subtracted from the third row.
2. Half of the second row is added to the third row.
3. The second row is multiplied by $-\frac12$.

The final matrix is upper triangular, so the system can be solved from the last row upward.

### Implementation

In [3]:
A = np.array([[1, 3, 1], [1, 1, -1], [3, 10, 5]], dtype=float)
b = np.array([9, 1, 32], dtype=float)

In [4]:
B = np.column_stack((A,b))
print(B)

[[ 1.  3.  1.  9.]
 [ 1.  1. -1.  1.]
 [ 3. 10.  5. 32.]]


In [5]:
def GE_swap(B,i,j):
    tmp = np.copy(B[i])
    B[i] = np.copy(B[j])
    B[j] = tmp

In [6]:
GE_swap(B,1,2)
print(B)
B = np.column_stack((A,b))

[[ 1.  3.  1.  9.]
 [ 3. 10.  5. 32.]
 [ 1.  1. -1.  1.]]


In [7]:
def GE_multiply(B,i,alpha):
    B[i] = B[i] * alpha

In [8]:
GE_multiply(B,0,0.5)
print(B)
B = np.column_stack((A,b))

[[ 0.5  1.5  0.5  4.5]
 [ 1.   1.  -1.   1. ]
 [ 3.  10.   5.  32. ]]


In [9]:
def GE_add_multiply(B,i,j,alpha):
    B[i] += B[j] * alpha

In [10]:
GE_add_multiply(B,1,0,-1)
print(B)

[[ 1.  3.  1.  9.]
 [ 0. -2. -2. -8.]
 [ 3. 10.  5. 32.]]


### Gauss elimination algorithm
1. Iterate over the columns, except for the last one ($i=0,\ldots,n-2$).
2. In column $i$, find a nonzero element in rows $i$ and below. For numerical stability, choose the one with the largest absolute value and swap its row with row $i$ if necessary.
3. Clear all elements in column $i$ below the pivot.

At the end we should have an upper-triangular system, which can be solved by back substitution. If no nonzero pivot remains in a column, the matrix is singular and the system must be checked for consistency and free variables.

In [11]:
#A safe point where we recreate the matrix and start our development
B = np.column_stack((A,b))
#GE_add_multiply(B,0,1,1)
#GE_swap(B,0,2)
print(B)
N = len(B)
print(N)

[[ 1.  3.  1.  9.]
 [ 1.  1. -1.  1.]
 [ 3. 10.  5. 32.]]
3


In [12]:
#column i from row i
i = 0
B[i:,i] 

array([1., 1., 3.])

In [13]:
pivot_tolerance = np.finfo(float).eps * N * np.linalg.norm(B[:, :N], ord=np.inf)
pivot_rows = np.flatnonzero(np.abs(B[i:, i]) > pivot_tolerance) + i
pivot_rows

array([0, 1, 2])

In [14]:
# Choose the largest-magnitude available pivot, not merely a positive entry.
if len(pivot_rows) == 0:
    raise np.linalg.LinAlgError("No usable pivot in this column")
j = i + int(np.argmax(np.abs(B[i:, i])))
print(j)

2


In [15]:
#if the nonzero is in an other row, swap the two
if i != j:
    GE_swap(B,i,j)
print(B)

[[ 3. 10.  5. 32.]
 [ 1.  1. -1.  1.]
 [ 1.  3.  1.  9.]]


In [16]:
# Normalize this pivot row, then eliminate the entries below its pivot.
GE_multiply(B, i, 1.0 / B[i, i])
for row in range(i + 1, N):
    GE_add_multiply(B, row, i, -B[row, i])
print(B)

[[ 1.          3.33333333  1.66666667 10.66666667]
 [ 0.         -2.33333333 -2.66666667 -9.66666667]
 [ 0.         -0.33333333 -0.66666667 -1.66666667]]


In [17]:
# Complete forward elimination with partial pivoting.
B = np.column_stack((A, b)).astype(float)
N = B.shape[0]
tolerance = np.finfo(float).eps * max(B.shape) * max(1.0, np.linalg.norm(B[:, :N], ord=np.inf))
for i in range(N):
    pivot_row = i + int(np.argmax(np.abs(B[i:, i])))
    if abs(B[pivot_row, i]) <= tolerance:
        raise np.linalg.LinAlgError("The coefficient matrix is singular")
    GE_swap(B, i, pivot_row)
    for row in range(i + 1, N):
        factor = B[row, i] / B[i, i]
        B[row, i:] -= factor * B[i, i:]
print(B)

[[ 3.         10.          5.         32.        ]
 [ 0.         -2.33333333 -2.66666667 -9.66666667]
 [ 0.          0.         -0.28571429 -0.28571429]]


In [18]:
# Back substitution: solve one equation at a time, from bottom to top.
x = np.zeros(N, dtype=float)
for i in reversed(range(N)):
    x[i] = (B[i, -1] - np.dot(B[i, i + 1:N], x[i + 1:N])) / B[i, i]
print(x)

[-1.  3.  1.]


## Solution

$$U =\begin{pmatrix}
1&3&1&9\\
0&1&1&4\\
0&0&1&1
\end{pmatrix}$$

Expanded form:
$$\begin{align}
x_3&=1\\
x_2&=4-x_3\\
x_1&=9-3x_2-x_3
\end{align}$$

For an upper-triangular system $U\mathbf{x}=\mathbf{c}$, back substitution is
$$x_i=\frac{c_i-\sum_{j=i+1}^{n}u_{ij}x_j}{u_{ii}},\qquad i=n,n-1,\ldots,1.$$
For this example, $\mathbf{x}=(-1,3,1)$. The diagonal entries must be nonzero; otherwise a unique solution cannot be found by this formula.

In [19]:
def my_GE(A, b):
    """Solve a nonsingular square system using Gaussian elimination."""
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)
    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        raise ValueError("A must be a square two-dimensional array")
    if b.shape != (A.shape[0],):
        raise ValueError("b must be a one-dimensional vector matching A")

    N = A.shape[0]
    B = np.column_stack((A, b))
    tolerance = np.finfo(float).eps * N * np.linalg.norm(A, ord=np.inf)

    for i in range(N):
        pivot_row = i + int(np.argmax(np.abs(B[i:, i])))
        if abs(B[pivot_row, i]) <= tolerance:
            raise np.linalg.LinAlgError(
                "A is singular to working precision; no unique solution exists"
            )
        GE_swap(B, i, pivot_row)
        for row in range(i + 1, N):
            factor = B[row, i] / B[i, i]
            B[row, i] = 0.0
            B[row, i + 1:] -= factor * B[i, i + 1:]

    solution = np.zeros(N, dtype=float)
    for i in reversed(range(N)):
        solution[i] = (
            B[i, -1] - np.dot(B[i, i + 1:N], solution[i + 1:N])
        ) / B[i, i]
    return solution

In [20]:
A = np.array([[1, 3, 1], [1, 1, -1], [3, 10, 5]], dtype=float)
b = np.array([9, 1, 32], dtype=float)
x = my_GE(A, b)
print(x)
print("Residual norm:", np.linalg.norm(A @ x - b))

[-1.  3.  1.]
Residual norm: 2.220446049250313e-16


In [21]:
A = np.array([[1,3,1],[1,1,-1],[3,10,5]],dtype=float)
b = np.array([9,1,32],dtype=float)
x = my_GE(A,b)
print(x)

[-1.  3.  1.]


## LU decomposition
The matrix $A$ is decomposed into a lower-triangular matrix $L$ and an upper-triangular matrix $U$ such that $A=LU$. Example:
$$
\begin{pmatrix}
a_{11} & a_{12} & a_{13} \\
a_{21} & a_{22} & a_{23} \\
a_{31} & a_{32} & a_{33}
\end{pmatrix}=
\begin{pmatrix}
l_{11} & 0 & 0 \\
l_{21} & l_{22} & 0 \\
l_{31} & l_{32} & l_{33}
\end{pmatrix}
\begin{pmatrix}
u_{11} & u_{12} & u_{13} \\
0 & u_{22} & u_{23} \\
0 & 0 & u_{33}
\end{pmatrix}.
$$
In particular, a common choice is to make the diagonal entries of $L$ equal to one:
$$L=\begin{pmatrix}
1 & 0 & 0 \\
l_{21} & 1 & 0 \\
l_{31} & l_{32} & 1
\end{pmatrix}.$$
With this convention, the unpivoted decomposition is unique when it exists.

#### Simple solution:
During Gaussian elimination, the multipliers used to clear entries below each pivot are recorded in $L$; the resulting upper-triangular matrix is $U$. To solve $A\mathbf{x}=\mathbf{b}$, first solve $L\mathbf{y}=\mathbf{b}$, then solve $U\mathbf{x}=\mathbf{y}$.

The entries of $U$ can be found by multiplying $L$ and $U$ and proceeding row by row. The first row gives
$$
\begin{align}
a_{11}&=u_{11}\\
a_{12}&=u_{12}\\
\vdots&=\vdots\\
a_{1n}&=u_{1n}
\end{align}
$$
so the first row of $U$ is the first row of $A$. The second row gives
$$
a_{21}=l_{21}u_{11},\qquad a_{2j}=l_{21}u_{1j}+u_{2j}\quad (j\ge2).
$$
Thus each step uses entries of $U$ that have already been found.

The algorithm must account for singular or small pivots by swapping rows and recording the swaps in a permutation matrix. This bookkeeping is why a library implementation is preferable to a short hand-written routine. SciPy returns factors satisfying $A=PLU$, equivalently $P^{\mathsf T}A=LU$. The routine below uses partial pivoting (row swaps); full pivoting also swaps columns.

In [22]:
from scipy.linalg import lu

A = np.array([[1, 3, 1], [1, 1, -1], [3, 10, 5]], dtype=float)
print(A)
P, L, U = lu(A)

[[ 1.  3.  1.]
 [ 1.  1. -1.]
 [ 3. 10.  5.]]


In [23]:
P,L,U

(array([[0., 0., 1.],
        [0., 1., 0.],
        [1., 0., 0.]]),
 array([[1.        , 0.        , 0.        ],
        [0.33333333, 1.        , 0.        ],
        [0.33333333, 0.14285714, 1.        ]]),
 array([[ 3.        , 10.        ,  5.        ],
        [ 0.        , -2.33333333, -2.66666667],
        [ 0.        ,  0.        , -0.28571429]]))

In [24]:
P.T @ A, L @ U

(array([[ 3., 10.,  5.],
        [ 1.,  1., -1.],
        [ 1.,  3.,  1.]]),
 array([[ 3., 10.,  5.],
        [ 1.,  1., -1.],
        [ 1.,  3.,  1.]]))

## Iterative methods: Gauss-Seidel
Sometimes iterative methods are desirable. If variables have very different scales, or if a system changes slowly over time so that successive solutions are close, an iterative method can be useful. Sparse matrices are another important case: iterative methods can take advantage of sparsity without forming a dense factorization.

Let us express each component of $\mathbf{x}$ from the system of linear equations, assuming we know the other components:
$$A\mathbf{x}=\mathbf{b}$$
In components:
$$
\begin{align}
a_{11}x_1+a_{12}x_2+\dots+a_{1n}x_n&=b_1\\
a_{21}x_1+a_{22}x_2+\dots+a_{2n}x_n&=b_2\\
\vdots&=\vdots\\
a_{n1}x_1+a_{n2}x_2+\dots+a_{nn}x_n&=b_n
\end{align}
$$
The components can be written iteratively using a superscript for the iteration count. First, the Jacobi method uses only values from the previous iteration:
$$\begin{align}
x_1^{(i+1)} &=\frac{-1}{a_{11}}(a_{12}x_2^{(i)}+\dots+a_{1n}x_n^{(i)}-b_1)\\
x_2^{(i+1)} &=\frac{-1}{a_{22}}(a_{21}x_1^{(i)}+a_{23}x_3^{(i)}+\dots+a_{2n}x_n^{(i)}-b_2)\\
\vdots&=\vdots\\
x_n^{(i+1)} &=\frac{-1}{a_{nn}}(a_{n1}x_1^{(i)}+a_{n2}x_2^{(i)}+\dots+a_{n,n-1}x_{n-1}^{(i)}-b_n)
\end{align}$$
Notice the minus sign multiplying the parentheses; it is equivalent to putting $b_j$ first in the numerator. Jacobi computes all new components from the old vector.

Gauss-Seidel uses the values already calculated in the current iteration. This often makes it converge faster, although convergence is not guaranteed for every matrix:
$$
\begin{align}
x_1^{(i+1)} &=\frac{-1}{a_{11}}(a_{12}x_2^{(i)}+\dots+a_{1n}x_n^{(i)}-b_1)\\
x_2^{(i+1)} &=\frac{-1}{a_{22}}(a_{21}x_1^{(i+1)}+a_{23}x_3^{(i)}+\dots+a_{2n}x_n^{(i)}-b_2)\\
\vdots&=\vdots\\
x_n^{(i+1)} &=\frac{-1}{a_{nn}}(a_{n1}x_1^{(i+1)}+a_{n2}x_2^{(i+1)}+\dots+a_{n,n-1}x_{n-1}^{(i+1)}-b_n)
\end{align}
$$
In sum form:
$$x_j^{(i+1)}=-\frac{1}{a_{jj}}\left(\sum_{k=1}^{j-1}a_{jk}x_k^{(i+1)}+\sum_{k=j+1}^{n}a_{jk}x_k^{(i)}-b_j\right).$$

Convergence is guaranteed when $A$ is strictly diagonally dominant by rows or symmetric positive definite. These are sufficient conditions; symmetry alone or positive diagonal entries alone are not enough. As with all iterative methods, a starting vector is required.

In [25]:
A = np.array([[10, -1, 2], [-1, 11, -1], [2, -1, 10]], dtype=float)
b = np.array([6, 22, -10], dtype=float)
N = len(b)
my_GE(A, b)

array([ 1.,  2., -1.])

In [26]:
#initial values
x0 = np.ones(N,dtype=float)
x = np.copy(x0)

In [27]:
# Update each component in place, reusing values computed earlier in this sweep.
for row in range(N):
    lower_sum = np.dot(A[row, :row], x[:row])
    upper_sum = np.dot(A[row, row + 1:], x[row + 1:])
    x[row] = (b[row] - lower_sum - upper_sum) / A[row, row]
    print(x)

[0.5 1.  1. ]
[0.5        2.13636364 1.        ]
[ 0.5         2.13636364 -0.88636364]


In [28]:
# A small update is useful to observe, but the residual checks the equations directly.
prev_x = np.copy(x)
for row in range(N):
    lower_sum = np.dot(A[row, :row], x[:row])
    upper_sum = np.dot(A[row, row + 1:], x[row + 1:])
    x[row] = (b[row] - lower_sum - upper_sum) / A[row, row]
print("Update norm:", np.linalg.norm(x - prev_x))
print("Residual norm:", np.linalg.norm(A @ x - b))

Update norm: 0.5190151553077321
Residual norm: 0.1459216356886062


In [29]:
def my_GS(A, b, lim=1e-8, max_iter=1000, x0=None):
    """Solve a square system approximately by Gauss-Seidel iteration.

    Returns the estimate, number of iterations, and final residual norm.
    """
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)
    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        raise ValueError("A must be a square two-dimensional array")
    if b.shape != (A.shape[0],):
        raise ValueError("b must be a one-dimensional vector matching A")
    if lim <= 0 or max_iter < 1:
        raise ValueError("lim and max_iter must be positive")

    N = A.shape[0]
    diagonal = np.diag(A)
    if np.any(diagonal == 0):
        raise ValueError("Gauss-Seidel requires nonzero diagonal entries")
    if x0 is None:
        x = np.ones(N, dtype=float)
    else:
        x = np.asarray(x0, dtype=float).copy()
        if x.shape != (N,):
            raise ValueError("x0 must be a vector matching the number of unknowns")

    for iteration in range(1, max_iter + 1):
        for row in range(N):
            lower_sum = np.dot(A[row, :row], x[:row])
            upper_sum = np.dot(A[row, row + 1:], x[row + 1:])
            x[row] = (b[row] - lower_sum - upper_sum) / diagonal[row]
        residual_norm = np.linalg.norm(A @ x - b)
        if residual_norm <= lim:
            break
    return x, iteration, residual_norm

In [30]:
A = np.array([[10, -1, 2], [-1, 11, -1], [2, -1, 10]], dtype=float)
b = np.array([6, 22, -10], dtype=float)
x, niter, residual = my_GS(A, b)
print(x)
print(niter, residual)

[ 1.  2. -1.]
8 1.3362764430660743e-09


In [31]:
%%timeit
A = np.array([[10, -1, 2], [-1, 11, -1], [2, -1, 10]], dtype=float)
b = np.array([6, 22, -10], dtype=float)
my_GE(A, b)

46.1 μs ± 3.33 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [32]:
%%timeit
A = np.array([[10, -1, 2], [-1, 11, -1], [2, -1, 10]], dtype=float)
b = np.array([6, 22, -10], dtype=float)
my_GS(A, b)

115 μs ± 7.16 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [33]:
# Symmetric positive-definite example that is not strictly diagonally dominant.
A = np.array([[4, 1, 1], [1, 1, 0], [1, 0, 1]], dtype=float)
b = np.array([5, 3, 0], dtype=float)
x, niter, residual = my_GS(A, b)
print(x)
print(niter, residual)

[ 1.  2. -1.]
27 7.450580596923828e-09
